In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

왜 거리 계산 metric에서 cosine을 많이쓸까?

- Cosine: 벡터 간 각도 측정
자연어 처리, 문서 검색, 추천 시스템 등 의미적 유사성이 중요한 경우

방향성 중심, 크기 문관

- 유클리드: 직선 거리 측정,  
k-means 클러스터링, 이미지 처리, 수치적 특성이 중요한 저차원 데이터의 경우

절대적 거리, 크기 민감

- 맨하탄: 각 차원 절댓값 차이 합
격자형 구조 데이터, 도시 블록 거리 계산 등 특수한 경우

L1, 격자 경로

- 고차원에서의 문제
텍스트 임베딩과 같은 고차원 공간에서는 차원의 저주 문제가 발생합니다. 차원이 높아질수록 모든 점들 간의 거리가 비슷해지는 경향이 있어, 유의미한 유사도 구분이 어려워집니다

- 코사인 유사도가 선호되는 이유
코사인 유사도는 벡터스토어에서 가장 널리 사용되는 이유가 방향성 기반 측정에 있다.. 텍스트 임베딩에서는 문서의 길이가 달라도 의미적으로 유사한 내용이면 비슷한 방향을 가리키기 때문이다다. 또한 -1에서 1 사이의 정규화된 값을 제공하여 일관된 척도로 비교가 가능하며 , 고차원 데이터에서 차원의 저주 문제를 완화한다.

- 유클리드/맨하탄 거리의 한계

유클리드 거리는 문서 길이에 민감하다는 치명적 단점이 있다. 같은 주제를 다루지만 길이가 다른 문서들이 유사도가 낮게 평가될 수 있으며, 고차원 공간에서는 모든 점들 간의 거리가 비슷해지는 문제가 발생한다. 맨하탄 거리는 계산이 빠르지만 의미적 유사성 측정에는 부적합하다.

# Qdrant Collection 생성

In [3]:
from qdrant_client import QdrantClient

client = QdrantClient(host="localhost", port=6333, prefer_grpc=True)

In [5]:
# 클라이언트 생성
from qdrant_client.http import models
coleect_name = "Image_RAG"

client.create_collection(
    collection_name=coleect_name,
    vectors_config=models.VectorParams(
        size=1024, # 벡터차원, bge m-3: 1024, openai embedding-3-small: 1536
        distance=models.Distance.COSINE # 거리 매트릭
    )
)

# 생성된 컬렉션 정보 확인
collecntion_info = client.get_collection(collection_name=coleect_name)

# 다중 벡터 컬렉션 생성

이미지 검색은 유클리드, text검색은 코사인등으로 설정

In [7]:

multi_vector_collection = "multi_vector_collection"

client.create_collection(
    collection_name=multi_vector_collection,
    vectors_config={
        "text": models.VectorParams(
            size=1024,
            distance=models.Distance.COSINE
        ),
        "image": models.VectorParams(
            size=512,
            distance=models.Distance.EUCLID
        )
    }
)

# 생성된 컬렉션 정보 확인
multi_vector_collection_info = client.get_collection(collection_name=multi_vector_collection)

In [ ]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode



# Qdant 고급설정 Collection 생성

In [ ]:
# 고급 설정이 적용된 컬렉션 생성
advanced_collection = "advanced_collection"

client.create_collection(
    collection_name=advanced_collection,
    vectors_config=models.VectorParams(
        size=384,
        distance=models.Distance.COSINE
    ),
    # HNSW 인덱스 설정
    hnsw_config=models.HnswConfigDiff(
        m=16,                     # 그래프의 각 노드에서 연결되는 이웃 수 (기본값: 16)
        ef_construct=100,         # 인덱스 구축 중 고려할 이웃 수 (기본값: 100)
        full_scan_threshold=10000, # 이 임계값 이하의 벡터 수에서는 전체 스캔 사용
        max_indexing_threads=0    # 0은 모든 사용 가능한 CPU 사용
    ),
    # 최적화 설정
    optimizers_config=models.OptimizersConfigDiff(
        deleted_threshold=0.2,    # 세그먼트 최적화 임계값 (삭제된 벡터 비율)
        vacuum_min_vector_number=1000,  # 최적화 최소 벡터 수
        default_segment_number=0, # 기본 세그먼트 수 (0은 자동)
        indexing_threshold=20000, # 인덱싱 임계값 (바이트)
        flush_interval_sec=5,     # 플러시 간격 (초)
        max_optimization_threads=1 # 최적화에 사용할 최대 스레드 수
    ),
    # 샤딩 설정 (분산 환경에서 유용)
    shard_number=1,
    # 온디스크 페이로드 설정 (메모리 사용량 최적화)
    on_disk_payload=False
)

